In [ ]:
import os
from dotenv import load_dotenv  # Importing the module to load environment variables

In [ ]:
# Load environment variables with error handling
try:
    load_dotenv()  # Loads environment variables from a .env file
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")  # Fetching the Google API key from environment
    if not GOOGLE_API_KEY:
        raise ValueError("GOOGLE_API_KEY not found in .env file")  # Raising an error if API key is not found
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY  # Setting the API key in the environment variables for usage
except Exception as e:
    print(f"Error loading environment variables: {str(e)}")  # If there is any error while loading, it will be printed
    raise

In [3]:
from langchain_community.document_loaders import PyPDFLoader  # Updated import

In [4]:
%pwd

'c:\\Users\\emon1\\Desktop\\Generative AI Mastery Full Course\\12 Project Advanced Exam QA Generator\\notebook'

In [5]:
%cd ..

c:\Users\emon1\Desktop\Generative AI Mastery Full Course\12 Project Advanced Exam QA Generator


In [ ]:
# PDF loading with validation
file_path = "data/Lecture-2.pdf"  # Specify the path to the PDF document
loader = PyPDFLoader(file_path)  # Initializing the loader
data = loader.load()  # Loading the content of the PDF

In [8]:
data

[Document(metadata={'producer': 'Skia/PDF m135 Google Apps Renderer', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/Lecture-2.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='What is an Embedded Computer?Embedded computers are everywhere. They are in phones, microwaves, airplanes, automobiles, calculators… The list goes on and on. An embedded computer, which is an integral component of most embedded systems, is a combination of hardware and software that is designated to perform a highly specific function. For example, the type of embedded computer in a washing machine will not be the same as the embedded computer in a Nikon camera. Because the software in embedded computers is designed to only execute certain tasks, the computer’s software in one device can be totally distinct from that of another. The hardware of an embedded computer is also specially designed to withstand stresses in its intended environment. For instance, an embedded computer installe

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Better for document processing

In [ ]:
# Extract content from the loaded PDF document
question_gen = ""
for page in data:
    question_gen += page.page_content  # Concatenate content from all the pages

In [10]:
question_gen

'What is an Embedded Computer?Embedded computers are everywhere. They are in phones, microwaves, airplanes, automobiles, calculators… The list goes on and on. An embedded computer, which is an integral component of most embedded systems, is a combination of hardware and software that is designated to perform a highly specific function. For example, the type of embedded computer in a washing machine will not be the same as the embedded computer in a Nikon camera. Because the software in embedded computers is designed to only execute certain tasks, the computer’s software in one device can be totally distinct from that of another. The hardware of an embedded computer is also specially designed to withstand stresses in its intended environment. For instance, an embedded computer installed under the hood of a car is designed to withstand high temperatures.Laptops and general-purpose CPU’s are not considered to be embedded computers for several reasons. First, the word “embedded” implies th

In [11]:
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Better for document processing

In [ ]:
# Text splitting with proper configuration
try:
    splitter_ques_gen = RecursiveCharacterTextSplitter(  # Initialize the text splitter
        chunk_size=1000,  # Set chunk size for the split
        chunk_overlap=100,  # Set overlap between chunks to maintain context
        separators=["\n\n", "\n", " ", ""]  # Define separators for splitting the text
    )
    chunk_ques_gen = splitter_ques_gen.split_text(question_gen)  # Split the content into chunks
    print(f"Split into {len(chunk_ques_gen)} chunks for question generation")  # Print the number of chunks created
except Exception as e:
    print(f"Error splitting text: {str(e)}")  # If there’s an error, display it
    raise

Split into 25 chunks for question generation


In [14]:
from langchain_core.documents import Document  # Updated import

In [ ]:
document_ques_gen = [Document(page_content=t) for t in chunk_ques_gen]  # Create Document objects from text chunks

In [ ]:
# Answer generation splitting
try:
    splitter_ans_gen = RecursiveCharacterTextSplitter(  # Initialize another splitter for answers
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    document_answer_gen = splitter_ans_gen.split_documents(document_ques_gen)  # Split the documents for answer generation
    print(f"Split into {len(document_answer_gen)} chunks for answer generation")  # Print the number of answer chunks created
except Exception as e:
    print(f"Error splitting documents: {str(e)}")  # If there’s an error, display it
    raise

Split into 25 chunks for answer generation


In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
try:
    llm_ques_gen_pipeline = ChatGoogleGenerativeAI(  # Initialize the Gemini model for question generation
        model="gemini-1.5-pro",  # Using the specific version of Gemini model
        temperature=0.3,  # Set the temperature for controlling randomness
        convert_system_message_to_human=True  # Ensure the system message is converted for human readability
    )
except Exception as e:
    print(f"Error initializing Gemini model: {str(e)}")  # If there’s an error, display it
    raise

In [26]:
from langchain_core.prompts import PromptTemplate

In [27]:
# Improved prompt templates
prompt_template = """
You are an expert at creating technical questions based on provided materials.
Your goal is to prepare developers for technical exams by creating relevant questions.

Text Content:
------------
{text}
------------

Instructions:
1. Create specific, technical questions covering key concepts
2. Include both conceptual and practical questions
3. Number each question clearly
4. Ensure questions are clear and unambiguous

Generated Questions:
"""

In [ ]:
# Create a prompt template for question generation
PROMPT_QUESTIONS = PromptTemplate(template=prompt_template, input_variables=['text'])

In [29]:
refine_template = """
You are refining technical questions based on additional context.
Existing Questions:
{existing_answer}

Additional Context:
------------
{text}
------------

Instructions:
1. Improve existing questions if needed
2. Add new questions if context reveals important information
3. Remove irrelevant questions
4. Maintain numbering and clarity

Refined Questions:
"""

In [ ]:
# Create a prompt template for refining existing questions
REFINE_PROMPT_QUESTIONS = PromptTemplate(
    input_variables=["existing_answer", "text"],
    template=refine_template,
)

In [31]:
from langchain.chains import load_summarize_chain

In [ ]:
# Generate questions using the defined chain
try:
    ques_gen_chain = load_summarize_chain(
        llm=llm_ques_gen_pipeline,  # Use the Gemini model for question generation
        chain_type="refine",  # Specify the type of chain to refine questions
        verbose=True,  # Enable verbose output for debugging
        question_prompt=PROMPT_QUESTIONS,  # Provide the prompt for question generation
        refine_prompt=REFINE_PROMPT_QUESTIONS  # Provide the prompt for refining questions
    )
    ques = ques_gen_chain.run(document_ques_gen)  # Run the chain on the documents
    print("Generated Questions:")
    print(ques)  # Display the generated questions
except Exception as e:
    print(f"Error generating questions: {str(e)}")  # If there's any error, display it
    raise

C:\Users\emon1\AppData\Local\Temp\ipykernel_11484\711232856.py:9: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  ques = ques_gen_chain.run(document_ques_gen)
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")




> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating technical questions based on provided materials.
Your goal is to prepare developers for technical exams by creating relevant questions.

Text Content:
------------
What is an Embedded Computer?Embedded computers are everywhere. They are in phones, microwaves, airplanes, automobiles, calculators… The list goes on and on. An embedded computer, which is an integral component of most embedded systems, is a combination of hardware and software that is designated to perform a highly specific function. For example, the type of embedded computer in a washing machine will not be the same as the embedded computer in a Nikon camera. Because the software in embedded computers is designed to only execute certain tasks, the computer’s software in one device can be totally distinct from that of another. The hardware of an embedded computer is also specially designe

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")



> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

You are refining technical questions based on additional context.
Existing Questions:
Refined Questions:

1.  What key characteristic distinguishes an embedded computer from a general-purpose computer like a laptop? Explain using the concept of "specific function" and the idea of being "contained within a larger system."

2.  Describe how the software of an embedded system differs from the software running on a desktop PC. Provide an example to illustrate this difference, focusing on the specialized nature of embedded software.

3.  Why is the hardware of an embedded computer often specialized?  Explain with reference to environmental considerations and the need to integrate with a larger system.

4.  True or False: An embedded computer's software is typically designed to be easily reconfigured for different tasks after deployment. Justify your answer, considering the "specific function" of embedded systems

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")



> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

You are refining technical questions based on additional context.
Existing Questions:
Refined Questions:

1.  What key characteristic distinguishes an embedded computer from a general-purpose computer like a laptop? Explain using the concept of "specific function" and the idea of being "contained within a larger system."

2.  Describe how the software of an embedded system differs from the software running on a desktop PC. Provide an example to illustrate this difference, focusing on the specialized nature of embedded software.

3.  Why is the hardware of an embedded computer often specialized?  Explain with reference to environmental considerations and the need to integrate with a larger system.

4.  True or False: An embedded computer's software is typically designed to be easily reconfigured for different tasks after deployment. Justify your answer, considering the "specific function" of embedded systems

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 51
}
].


Error generating questions: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 48
}
]


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 48
}
]